# 3-4 体裁を整える（日本語フォント・配色・軸の書式）

ここまでで「グラフを描く」ところまで来ました。最後にやるのは **「人に見せられる体裁にする」** 仕上げの工程です。

デフォルトのままだと、軸が `100000` のような **生の数字**、配色が **ピンと来ない既定色**、タイトルが **小さくて素っ気ない** …といったところが気になります。本章ではそれらを **数行で直す型** を覚えていきます。

## このノートのゴール

- **日本語フォント** が文字化け（豆腐）にならない仕組みを理解する
- **3桁区切り** や **「万円」表示** で軸を読みやすくする
- **配色** を 1 行で切り替えて、レポート向けに整える
- **タイトル・余白** を整え、`savefig` で **画像ファイル** として保存する

## Excel との対比

| やりたいこと | Excel | matplotlib |
|---|---|---|
| 軸を「桁区切り」表示に | 軸を右クリック → 表示形式 → `#,##0` | `ax.yaxis.set_major_formatter(...)` |
| 「万円」表示に | 軸の表示形式 → `#,##0,,"万円"` 等 | `FuncFormatter(lambda x, _: f"{x/10000:,.0f}")` |
| 単位を軸の上に小さく置く | テキストボックスで「(万円)」を軸の上に | `ax.text(0, 1.02, "(万円)", transform=ax.transAxes)` |
| 色を変える | 系列を右クリック → 塗りつぶしの色 | `color="..."` または `colormap="..."` |
| グラフを画像で保存 | グラフを右クリック → 画像として保存 | `plt.savefig("...png")` |

Excel だと **クリックを重ねて** 整えるところを、matplotlib は **行を 1〜2 本足すだけ** で済ませられます。一度書けば、来月も再来月も **同じ体裁** で出力できるのが強みです。

## 準備

3-1 〜 3-3 と同じ読み込みです。ここからは `daily` `by_product` `pivot` の 3 つを軸に体裁を整えていきます。

In [ ]:
!pip install -q japanize-matplotlib

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import japanize_matplotlib
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/python-data-basics"
DATA = f"{BASE}/data/pandas"

df = pd.read_excel(f"{DATA}/sales_2026-01.xlsx", sheet_name="売上", parse_dates=["date"])

daily = df.groupby("date")["amount"].sum()
by_product = df.groupby("product_name")["amount"].sum().sort_values(ascending=False)
pivot = df.pivot_table(index="date", columns="product_name", values="amount", aggfunc="sum", fill_value=0)

df.head()

## 1. 日本語フォント — `japanize-matplotlib`

matplotlib は **標準のままだと日本語が文字化け（□□□ のような豆腐）** します。日本語フォントが入っていないからです。

解決は 2 行だけ:

```python
!pip install -q japanize-matplotlib
import japanize_matplotlib
```

**import した時点で**、matplotlib のデフォルトフォントが日本語対応のものに差し替わります。以後 `plt.title("売上")` などがそのまま使えます。

> 💡 上のセルですでに import 済みなので、このノート内では何も追加しなくても日本語が表示されます。

In [ ]:
# 日本語タイトルが豆腐にならないことを確認
by_product.plot.bar(figsize=(9, 4), title="商品別 売上合計", color="steelblue")
plt.show()

## 2. 軸の数値書式 — 3桁区切りにする

デフォルトだと y 軸が `100000` `200000` のように **カンマなしの生の数字** で出ます。読みづらいので **3桁区切り** にしましょう。

型はこれです:

```python
ax = グラフを描く
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
```

- `.plot(...)` は **`ax` というオブジェクト** を返します。これに対して軸の書式を指示します。
- `"{x:,.0f}"` は Python の **`f"{値:,.0f}"`** と同じ書式指定です（カンマ区切り・小数 0 桁）。

### 単位（円・万円など）は軸の上に小さく置く

matplotlib の `plt.ylabel("売上 (万円)")` は **既定で 90 度回転** して縦倒しの日本語になり、ひどく読みづらくなります。Excel のグラフのように **軸の数値の真上に小さく横書き** で置くのが見やすい。

これからの作例では、`ylabel` は使わず、

```python
ax.text(0, 1.02, "(円)", transform=ax.transAxes, ha="left", fontsize=10)
```

で軸の上に出すパターンを使います。`transform=ax.transAxes` は「軸を 0〜1 の座標系として扱う」指定で、グラフの大きさや値が変わっても **常に同じ位置** に出ます。

In [ ]:
ax = daily.plot(figsize=(10, 4), title="日次売上推移", marker="o")
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))

# 単位は軸の上に小さく
ax.text(0, 1.02, "(円)", transform=ax.transAxes, ha="left", fontsize=10)

plt.grid(True, alpha=0.3)
plt.show()

## 3. 「万円」表示に変える

売上が大きくなると、円のままでは桁が多すぎて読みにくくなります。**「万円」単位** に変えてみましょう。

やり方は **値を 10000 で割って表示** するだけ。`FuncFormatter` で自分用の関数を渡せます。

```python
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
```

> `lambda x, _: ...` の `_` は「使わない引数」の慣習的な書き方です（matplotlib が `value, position` の 2 引数で呼んでくるので、形を合わせています）。

軸の数値を割り算したら、**軸の上の単位ラベルも合わせて変える** のを忘れずに。

In [ ]:
ax = daily.plot(figsize=(10, 4), title="日次売上推移", marker="o")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)

plt.grid(True, alpha=0.3)
plt.show()

## 4. 配色 — 1 色指定とカラーパレット

色の指定は 2 通りあります。

**単一系列なら `color=` で 1 色指定**:

```python
by_product.plot.bar(color="steelblue")
```

**複数系列なら `colormap=` でパレットを当てる** のが手軽です:

```python
pivot.plot.bar(stacked=True, colormap="tab20")
```

よく使うパレット名:

| `colormap` | 雰囲気 | 用途 |
|---|---|---|
| `"tab10"` / `"tab20"` | はっきりした原色 | カテゴリ分け |
| `"Pastel1"` / `"Pastel2"` | やわらかい色 | レポート向け |
| `"viridis"` / `"plasma"` | 連続的なグラデーション | 大小に意味があるとき |
| `"Set2"` | 落ち着いた中間色 | プレゼン資料 |

In [ ]:
# 複数系列をパレットで色分け
ax = pivot.plot.bar(stacked=True, figsize=(14, 5), colormap="Pastel1", title="日次売上の商品別内訳")
ax.text(0, 1.02, "(円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

## 5. タイトル・余白の仕上げ

もう少しだけ整えます。

- **タイトルを大きく・太く**: `plt.title("...", fontsize=14, fontweight="bold")`
- **軸の上の単位を少し大きく**: `ax.text(..., fontsize=11)`
- **余白がはみ出さないように**: `plt.tight_layout()`

`tight_layout()` は **「ラベルやタイトルが切れないように余白を自動調整して」** という指示です。`plt.show()` や `plt.savefig()` の **直前に 1 回** 呼べば OK。

In [ ]:
ax = by_product.plot.bar(figsize=(9, 4), color="steelblue")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("商品別 売上合計", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=11)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. 画像ファイルとして保存する — `savefig`

整えたグラフは **PNG ファイル** として保存して、後で資料に貼り付けられます。

```python
plt.savefig("保存先のパス.png", dpi=150, bbox_inches="tight")
```

- **`dpi=150`**: 解像度。150 程度あれば資料用に十分きれい
- **`bbox_inches="tight"`**: 余白を最小限にカット
- **`plt.show()` の前** に `savefig` を呼ぶこと。`show()` の後だと **空の画像** が保存されることがあります

保存先は Google Drive の中。第4章の月次レポートでも同じやり方で出力します。

In [ ]:
import os
OUT = f"{BASE}/output"
os.makedirs(OUT, exist_ok=True)   # フォルダが無ければ作る

ax = by_product.plot.bar(figsize=(9, 4), color="steelblue")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("商品別 売上合計", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=11)
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(f"{OUT}/by_product.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"保存しました → {OUT}/by_product.png")

## チェックリスト — 「レポートに耐える」見た目に

毎回これを確認すれば、ぐっと印象が変わります。

- [ ] 日本語が **豆腐** になっていないか（`import japanize_matplotlib`）
- [ ] 軸の数値が **3桁区切り or 万円表示** になっているか
- [ ] **単位 (円・万円など) が縦倒しの y ラベルではなく、軸の上に横書きで** 置かれているか
- [ ] タイトルが **「何のグラフか」 一目で分かる** か
- [ ] 配色は **多すぎず・派手すぎず**（`colormap="Pastel1"` `"Set2"` 等）
- [ ] 凡例がグラフに **被っていない** か（`bbox_to_anchor=(1, 1)`）
- [ ] `plt.tight_layout()` で **ラベルが切れていない** か
- [ ] `savefig` で **画像保存** までできているか

## 練習問題

本文のコードを少しだけ変える形の演習です。`daily` `by_product` `pivot` `OUT` はそのまま使ってください。

1. **`pivot` の積み上げ棒グラフ** を、配色 **`"Set2"`** で描いてください。本文 4 章の `colormap="Pastel1"` を **`"Set2"` に変える** だけ。
2. **`daily` の折れ線** で、y 軸を **「千円」** 表示にしてください。本文 3 章の **`x/10000` を `x/1000` に変え**、軸の上の **`"(万円)"` を `"(千円)"` に変える** だけ。
3. **`pivot` の積み上げ棒** を **`stacked.png`** という名前で保存してください。本文 6 章の `by_product` の描画を **`pivot.plot.bar(stacked=True, ...)`** に差し替え、保存ファイル名を変えるだけ。

In [ ]:
# ここに書いてみてください


<details>
<summary>答えを見る</summary>

```python
# 1. pivot を Set2 配色で
ax = pivot.plot.bar(stacked=True, figsize=(14, 5), colormap="Set2", title="日次売上の商品別内訳")
ax.text(0, 1.02, "(円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

# 2. daily を「千円」表示の折れ線で
ax = daily.plot(figsize=(10, 4), title="日次売上推移", marker="o")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/1000:,.0f}")
)
ax.text(0, 1.02, "(千円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()

# 3. pivot の積み上げ棒を stacked.png で保存
ax = pivot.plot.bar(stacked=True, figsize=(14, 5), colormap="Pastel1", title="日次売上の商品別内訳")
ax.text(0, 1.02, "(円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

plt.savefig(f"{OUT}/stacked.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"保存しました → {OUT}/stacked.png")
```

</details>

## よくあるエラー

### 1. 日本語が □□□ のままになる
→ `import japanize_matplotlib` のセルを実行していない、または **import したあとに `plt.rcParams` を上書き** している。`japanize_matplotlib` は **import するだけ** で効くので、特別な呼び出しは不要。

### 2. y 軸ラベルが **縦倒し** で読みづらい
→ `plt.ylabel("売上 (万円)")` は **既定で 90 度回転** する。`ylabel` は使わず、`ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)` で **軸の上に横書きで** 単位を置く。

### 3. `set_major_formatter` で `NameError: ticker is not defined`
→ 冒頭の `import matplotlib.ticker as ticker` が抜けている。`plt` とは別に **`ticker` も import** する。

### 4. `savefig` したのに **空っぽの白い画像** が保存される
→ `plt.show()` の **後ろ** で `savefig` を呼んでいる。`show()` を呼ぶと内部のグラフがリセットされるので、**`savefig` → `show`** の順番にする。

### 5. 画像の **端が切れている**
→ `bbox_inches="tight"` を付け忘れ。または `plt.tight_layout()` を呼んでいない。両方やっておくと安心。

### 6. `os.makedirs` で `FileExistsError`
→ **`exist_ok=True`** を付け忘れ。これを付けると、フォルダがすでにあってもエラーにならない。

## まとめ

- **`japanize_matplotlib`** を import するだけで日本語が通る
- 軸の数値書式は **`ax.yaxis.set_major_formatter(...)`** で 3 桁区切り・万円表示
- 単位 (円・万円) は **縦倒しの `ylabel` ではなく、`ax.text(0, 1.02, "(万円)", transform=ax.transAxes)` で軸の上に横書き**
- 配色は **`color="..."`** か **`colormap="..."`**、レポート向けは `Pastel1` / `Set2`
- 仕上げは **`plt.tight_layout()`**、保存は **`plt.savefig(..., dpi=150, bbox_inches="tight")`**
- **`savefig` → `show`** の順番を守る

これで第 3 章は終わりです。`pandas` での集計と `matplotlib` での可視化、そして体裁の整え方が揃いました。

次の **第 4 章「月次売上レポート自動生成」** では、ここまでの内容を **全部つないで** 、**複数月分の Excel を読み込み → 集計 → グラフ → Excel と画像で書き出す** という、**実務に直結する 1 本のスクリプト** を組み立てていきます。